# P2-A6 — Build an Agent (the multi-step loop)

In P2-A5 you ran *one* tool call by hand. An **agent** is that loop, automated and repeated: the model calls a tool, you run it and feed the result back, and it decides what to do next — until the task is done. You'll build the loop you described in P2-A5's writeup, with **multiple tools** so the model has to *chain* them.

Task: *"What's the status of my most recent order?"* requires two steps — `list_orders` (find the most recent), then `get_order_status` on that ID. The agent figures out that sequence itself.

In [ ]:
# --- Setup (given) ---
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o-mini'

# --- A mini order system with THREE tools ---
ORDERS = {
    'A123': {'status': 'shipped',    'eta_days': 2, 'placed': '2026-06-20'},
    'B456': {'status': 'processing', 'eta_days': 5, 'placed': '2026-06-27'},
}

def list_orders():
    """Return all orders with their placed-date (most recent has the latest date)."""
    return [{'order_id': k, 'placed': v['placed']} for k, v in ORDERS.items()]

def get_order_status(order_id):
    return ORDERS.get(order_id, {'status': 'not_found'})

def cancel_order(order_id):
    if order_id in ORDERS:
        ORDERS[order_id]['status'] = 'cancelled'
        return {'order_id': order_id, 'result': 'cancelled'}
    return {'order_id': order_id, 'result': 'not_found'}

TOOL_REGISTRY = {
    'list_orders': list_orders,
    'get_order_status': get_order_status,
    'cancel_order': cancel_order,
}

tools = [
    {'type': 'function', 'function': {
        'name': 'list_orders', 'description': 'List all of the customer\u2019s orders with the date each was placed.',
        'parameters': {'type': 'object', 'properties': {}, 'required': []}}},
    {'type': 'function', 'function': {
        'name': 'get_order_status', 'description': 'Get the status and ETA of one order by its ID.',
        'parameters': {'type': 'object', 'properties': {'order_id': {'type': 'string'}}, 'required': ['order_id']}}},
    {'type': 'function', 'function': {
        'name': 'cancel_order', 'description': 'Cancel one order by its ID.',
        'parameters': {'type': 'object', 'properties': {'order_id': {'type': 'string'}}, 'required': ['order_id']}}},
]
print('tools:', list(TOOL_REGISTRY))

## Task 1 — Write the agent loop
Write `run_agent(user_message, max_steps=5)` that:
1. Starts `messages` with the user message.
2. Loops (up to `max_steps`): call the model with `tools=tools`.
3. If the reply has **no** `tool_calls` → return its text (done).
4. If it **does** → append the assistant message, then for **each** tool call: dispatch `TOOL_REGISTRY[name](**args)`, append a `role='tool'` message with the result. Then loop again.
5. `print` each tool the agent calls, so you can see its reasoning trace.
<br>Hint: a single reply can contain *multiple* `tool_calls` — iterate over all of them before re-calling. The `max_steps` cap prevents an infinite loop.

In [ ]:
# TODO: def run_agent(user_message, max_steps=5): ... the loop


## Task 2 — A task that needs chaining
Run `run_agent("What's the status of my most recent order?")`. Watch the trace: it should call `list_orders` first, then `get_order_status` on `B456` (the most recent), then answer.
<br>Goal: you didn't tell it the steps — it *planned* them from the tools available.

In [ ]:
# TODO: run the agent on the multi-step status question


## Task 3 — An action that changes state
Run `run_agent("Please cancel my most recent order.")`, then check `ORDERS` to confirm `B456` is now `cancelled`.
<br>Goal: the agent chained `list_orders` → `cancel_order` and actually *did* something. (Notice: a real system would confirm before a destructive action — note that in Task 4.)

In [ ]:
# TODO: run the agent on the cancel task; then print ORDERS to verify the change


## Task 4 — Explain (in your own words)
1. What specifically makes this an *agent* rather than the single tool call from P2-A5?
2. Autonomous loops that take real actions are powerful and risky. Name two things you'd add before letting this run against a real order system (you already hinted at one in P2-A5).

> _your answer here_